In [1]:
pip install --force-reinstall numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 131.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.25.2
    Uninstalling numpy-1.25.2:
      Successfully uninstalled numpy-1.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.9.3 requires numpy<1.26.0,>=1.18.5, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
"""
ABLATION STUDY VISUALISATION — WITH 95% CI + BAR LABELS RESTORED
===============================================================
Now keeps ALL conditions, but forces heatmap row order:
 → "country_only" at TOP
 → "full" at BOTTOM
 → everything else stays in between
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style("white")
plt.rcParams.update({'figure.dpi':300,'savefig.dpi':300,'font.size':10,'axes.grid':False})


# ============================================================
# BOOTSTRAP MAE + 95% CI
# ============================================================
def calculate_metrics_with_ci(subset, n_bootstrap=800, confidence=0.95):

    if len(subset)==0:
        return {'mae':np.nan,'mae_lower':np.nan,'mae_upper':np.nan,'rmse':np.nan,'r2':np.nan,'n':0}

    mae = mean_absolute_error(subset.actual_other_willingness,subset.prediction)
    rmse= np.sqrt(mean_squared_error(subset.actual_other_willingness,subset.prediction))
    r2  = r2_score(subset.actual_other_willingness,subset.prediction)

    boot=[]
    idx=np.arange(len(subset))
    np.random.seed(42)
    for _ in range(n_bootstrap):
        pick=np.random.choice(idx,size=len(idx),replace=True)
        boot.append(mean_absolute_error(subset.actual_other_willingness.iloc[pick],
                                        subset.prediction.iloc[pick]))

    lo=np.percentile(boot,2.5)
    hi=np.percentile(boot,97.5)

    return {'mae':mae,'mae_lower':lo,'mae_upper':hi,'rmse':rmse,'r2':r2,'n':len(subset)}


# ============================================================
# LOAD + REMOVE COUNTERFACTUALS
# ============================================================
def load_data(file,label):
    print(f"\n📄 Loading {file} [{label}]")
    df=pd.read_csv(file)
    df=df.dropna(subset=['prediction','actual_other_willingness'])
    df.actual_other_willingness*=100; df.actual_own_willingness*=100

    # remove CF trials
    df=df[~df.condition.isin(['cf_gdp_flip','cf_name_mismatch','cf_willingness_flip'])]
    print(f"  ✓ {len(df)} rows retained")
    return df


# ============================================================
# HEATMAP — MAE ± CI  (Country Only → middle → Full)
# ============================================================
def heatmap(df,metrics,out,title):

    models = sorted(df.model.unique())

    # 🔥 full dynamic ordering
    all_conds = sorted(df.condition.unique())
    conds = (["country_only"] +
            [c for c in all_conds if c not in ["country_only","full"]] +
            ["full"])

    mat = np.zeros((len(conds),len(models))) * np.nan
    ann = np.empty((len(conds),len(models)),dtype=object)

    for i,c in enumerate(conds):
        for j,m in enumerate(models):
            if (c,m) in metrics:
                d  = metrics[(c,m)]
                ci = (d['mae_upper']-d['mae_lower'])/2
                mat[i,j]=d['mae']
                ann[i,j]=f"{d['mae']:.1f}±{ci:.1f}"

    fig,ax=plt.subplots(figsize=(10,8))
    sns.heatmap(mat,annot=ann,fmt="",cmap="RdYlGn_r",
        xticklabels=[m.upper() for m in models],
        yticklabels=[c.replace('_',' ') for c in conds],
        cbar_kws={'label':'MAE (pp)'},linewidths=.6)

    ax.set_title(f"{title} — MAE ± 95% CI",fontweight="bold")
    plt.tight_layout(); plt.savefig(out); plt.savefig(out.replace(".png",".pdf")); plt.close()


# ============================================================
# 4-PANEL ΔMAE (unchanged)
# ============================================================
def panel(df,metrics,baseline,out,title):

    models=sorted(df.model.unique())
    conds=[c for c in sorted(df.condition.unique()) if c!="full"]

    fig,axs=plt.subplots(2,2,figsize=(16,12)); axs=axs.flatten()

    for i,m in enumerate(models):

        ax=axs[i]; vals={}; err=[]
        for c in conds:
            if (c,m) in metrics:
                d=metrics[(c,m)]; b=baseline[m]['mae']
                mean=d['mae']-b
                lo  =d['mae_lower']-b
                hi  =d['mae_upper']-b
                vals[c]=mean; err.append([mean-lo,hi-mean])

        ks=sorted(vals,key=lambda x:vals[x])
        y=np.arange(len(ks)); v=[vals[k] for k in ks]

        bars=ax.barh(y,v,
            color=['#d62728' if x>0 else '#2ca02c' for x in v],
            edgecolor="black",alpha=.75)

        ax.errorbar(v,y,xerr=np.array(err).T,fmt="none",
                    ecolor="black",capsize=4,linewidth=1.2)

        for bar,val in zip(bars,v):
            ax.text(val+(0.3 if val>0 else -0.3),
                    bar.get_y()+bar.get_height()/2,
                    f"{val:+.2f}",
                    ha='left' if val>0 else 'right',
                    va='center',fontweight="bold")

        ax.axvline(0,color="black")
        ax.set_yticks(y)
        ax.set_yticklabels([k.replace('_',' ') for k in ks])
        ax.set_title(f"{m.upper()} baseline {baseline[m]['mae']:.2f}pp")
        ax.set_xlabel("Δ MAE (pp)")

    plt.suptitle(f"{title} — ΔMAE ±95% CI",fontweight="bold")
    plt.tight_layout(); plt.savefig(out); plt.savefig(out.replace(".png",".pdf")); plt.close()


# ============================================================
# RUN PIPELINE
# ============================================================
def run(file,label,prefix):

    df=load_data(file,label)

    models=sorted(df.model.unique())
    conds =sorted(df.condition.unique())
    baseline={m:calculate_metrics_with_ci(df[(df.model==m)&(df.condition=='full')]) for m in models}
    metrics={(c,m):calculate_metrics_with_ci(df[(df.model==m)&(df.condition==c)]) for c in conds for m in models}

    heatmap(df,metrics,f"{prefix}_mae_heatmap.png",label)
    panel   (df,metrics,baseline,f"{prefix}_degradation_4.png",label)


if __name__=="__main__":
    run("ablation_original_raw_results.csv","ORIGINAL","original")
    run("ablation_structured_climate_raw_results.csv","STRUCTURED","structured")
    print("\n✔ Ablation visualisation complete — country_only top / full bottom\n")



📄 Loading ablation_original_raw_results.csv [ORIGINAL]
  ✓ 3500 rows retained

📄 Loading ablation_structured_climate_raw_results.csv [STRUCTURED]
  ✓ 3500 rows retained

✔ Ablation visualisation complete — country_only top / full bottom



In [7]:
pip install openpyxl --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 23.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
"""
ABLATION STUDY — COMPREHENSIVE RESULTS TABLE WITH 95% CI
=========================================================
Produces a detailed table showing MAE, RMSE, R² with 95% CI
for all models across all conditions (including counterfactuals).
"""

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')



# ============================================================
# BOOTSTRAP METRICS + 95% CI
# ============================================================
def calculate_all_metrics_with_ci(subset, n_bootstrap=800, confidence=0.95):
    """
    Calculate MAE, RMSE, R² with 95% confidence intervals via bootstrap
    """
    if len(subset) == 0:
        return {
            'n': 0,
            'mae': np.nan, 'mae_lower': np.nan, 'mae_upper': np.nan,
            'rmse': np.nan, 'rmse_lower': np.nan, 'rmse_upper': np.nan,
            'r2': np.nan, 'r2_lower': np.nan, 'r2_upper': np.nan
        }
    
    # Point estimates
    mae = mean_absolute_error(subset.actual_other_willingness, subset.prediction)
    rmse = np.sqrt(mean_squared_error(subset.actual_other_willingness, subset.prediction))
    r2 = r2_score(subset.actual_other_willingness, subset.prediction)
    
    # Bootstrap
    boot_mae = []
    boot_rmse = []
    boot_r2 = []
    
    idx = np.arange(len(subset))
    np.random.seed(42)
    
    for _ in range(n_bootstrap):
        pick = np.random.choice(idx, size=len(idx), replace=True)
        y_true = subset.actual_other_willingness.iloc[pick]
        y_pred = subset.prediction.iloc[pick]
        
        boot_mae.append(mean_absolute_error(y_true, y_pred))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true, y_pred)))
        boot_r2.append(r2_score(y_true, y_pred))
    
    # Confidence intervals
    alpha = (1 - confidence) / 2 * 100
    
    return {
        'n': len(subset),
        'mae': mae,
        'mae_lower': np.percentile(boot_mae, alpha),
        'mae_upper': np.percentile(boot_mae, 100 - alpha),
        'rmse': rmse,
        'rmse_lower': np.percentile(boot_rmse, alpha),
        'rmse_upper': np.percentile(boot_rmse, 100 - alpha),
        'r2': r2,
        'r2_lower': np.percentile(boot_r2, alpha),
        'r2_upper': np.percentile(boot_r2, 100 - alpha)
    }


# ============================================================
# LOAD DATA
# ============================================================
def load_data(file, label, remove_counterfactuals=True):
    print(f"\n📄 Loading {file} [{label}]")
    df = pd.read_csv(file)
    df = df.dropna(subset=['prediction', 'actual_other_willingness'])
    df.actual_other_willingness *= 100
    df.actual_own_willingness *= 100
    
    if remove_counterfactuals:
        df = df[~df.condition.isin(['cf_gdp_flip', 'cf_name_mismatch', 'cf_willingness_flip'])]
        print(f"  ✓ {len(df)} rows retained (counterfactuals removed)")
    else:
        print(f"  ✓ {len(df)} rows retained (all conditions included)")
    
    return df


# ============================================================
# CREATE RESULTS TABLE
# ============================================================
def create_results_table(df, label):
    """
    Create comprehensive results table with all metrics and 95% CI
    """
    models = sorted(df.model.unique())
    conditions = sorted(df.condition.unique())
    
    results = []
    
    for condition in conditions:
        for model in models:
            subset = df[(df.model == model) & (df.condition == condition)]
            metrics = calculate_all_metrics_with_ci(subset)
            
            results.append({
                'Condition': condition,
                'Model': model.upper(),
                'N': metrics['n'],
                'MAE': f"{metrics['mae']:.2f}",
                'MAE_CI': f"[{metrics['mae_lower']:.2f}, {metrics['mae_upper']:.2f}]",
                'MAE_±': f"±{(metrics['mae_upper']-metrics['mae_lower'])/2:.2f}",
                'RMSE': f"{metrics['rmse']:.2f}",
                'RMSE_CI': f"[{metrics['rmse_lower']:.2f}, {metrics['rmse_upper']:.2f}]",
                'RMSE_±': f"±{(metrics['rmse_upper']-metrics['rmse_lower'])/2:.2f}",
                'R²': f"{metrics['r2']:.3f}",
                'R²_CI': f"[{metrics['r2_lower']:.3f}, {metrics['r2_upper']:.3f}]",
                'R²_±': f"±{(metrics['r2_upper']-metrics['r2_lower'])/2:.3f}"
            })
    
    results_df = pd.DataFrame(results)
    
    # Sort: country_only first, full last, alphabetical in between
    condition_order = (
        ['country_only'] +
        sorted([c for c in conditions if c not in ['country_only', 'full']]) +
        ['full']
    )
    
    results_df['Condition'] = pd.Categorical(
        results_df['Condition'],
        categories=condition_order,
        ordered=True
    )
    results_df = results_df.sort_values(['Condition', 'Model'])
    
    return results_df


# ============================================================
# CREATE COMPACT TABLE (MAE only with ±)
# ============================================================
def create_compact_table(df, label):
    """
    Create compact table showing just MAE ± CI for easy reading
    """
    models = sorted(df.model.unique())
    conditions = sorted(df.condition.unique())
    
    # Reorder conditions
    condition_order = (
        ['country_only'] +
        sorted([c for c in conditions if c not in ['country_only', 'full']]) +
        ['full']
    )
    
    results = []
    
    for condition in condition_order:
        row = {'Condition': condition.replace('_', ' ').title()}
        for model in models:
            subset = df[(df.model == model) & (df.condition == condition)]
            metrics = calculate_all_metrics_with_ci(subset)
            
            mae = metrics['mae']
            ci = (metrics['mae_upper'] - metrics['mae_lower']) / 2
            
            row[model.upper()] = f"{mae:.2f}±{ci:.2f}"
        
        results.append(row)
    
    return pd.DataFrame(results)


# ============================================================
# MAIN PIPELINE
# ============================================================
def run_ablation_tables(file, label, prefix, include_counterfactuals=False):
    """
    Generate comprehensive ablation results tables
    """
    df = load_data(file, label, remove_counterfactuals=not include_counterfactuals)
    
    # Full results table
    full_table = create_results_table(df, label)
    full_table.to_csv(f"{prefix}_ablation_full_results.csv", index=False)
    print(f"\n✅ Saved: {prefix}_ablation_full_results.csv")
    
    # Compact table (MAE only)
    compact_table = create_compact_table(df, label)
    compact_table.to_csv(f"{prefix}_ablation_compact_results.csv", index=False)
    print(f"✅ Saved: {prefix}_ablation_compact_results.csv")
    
    # Also save as Excel for easier viewing
    with pd.ExcelWriter(f"{prefix}_ablation_results.xlsx") as writer:
        full_table.to_excel(writer, sheet_name='Full Results', index=False)
        compact_table.to_excel(writer, sheet_name='Compact (MAE)', index=False)
    
    print(f"✅ Saved: {prefix}_ablation_results.xlsx")
    
    return full_table, compact_table


# ============================================================
# RUN FOR BOTH DATASETS
# ============================================================
if __name__ == "__main__":
    
    print("\n" + "="*60)
    print("ORIGINAL DATASET")
    print("="*60)
    orig_full, orig_compact = run_ablation_tables(
        "ablation_original_raw_results.csv",
        "ORIGINAL",
        "original"
    )
    
    print("\n" + "="*60)
    print("STRUCTURED DATASET")
    print("="*60)
    struct_full, struct_compact = run_ablation_tables(
        "ablation_structured_climate_raw_results.csv",
        "STRUCTURED",
        "structured"
    )
    
    print("\n" + "="*60)
    print("PREVIEW: Original Compact Table")
    print("="*60)
    print(orig_compact.to_string(index=False))
    
    print("\n" + "="*60)
    print("PREVIEW: Structured Compact Table")
    print("="*60)
    print(struct_compact.to_string(index=False))
    
    print("\n✔ All ablation results tables generated successfully\n")


ORIGINAL DATASET

📄 Loading ablation_original_raw_results.csv [ORIGINAL]
  ✓ 3500 rows retained (counterfactuals removed)

✅ Saved: original_ablation_full_results.csv
✅ Saved: original_ablation_compact_results.csv
✅ Saved: original_ablation_results.xlsx

STRUCTURED DATASET

📄 Loading ablation_structured_climate_raw_results.csv [STRUCTURED]
  ✓ 3500 rows retained (counterfactuals removed)

✅ Saved: structured_ablation_full_results.csv
✅ Saved: structured_ablation_compact_results.csv
✅ Saved: structured_ablation_results.xlsx

PREVIEW: Original Compact Table
         Condition     CLAUDE     GEMINI        GPT      LLAMA
      Country Only 16.04±1.50 17.62±1.85 16.09±1.28 12.27±1.28
        No Climate  4.66±0.66 14.77±1.42 12.41±1.08  6.05±0.71
           No Demo  5.11±0.67 17.01±1.51 16.20±1.39  7.82±1.00
           No Econ  4.92±0.66 17.12±1.47 14.57±1.14  6.62±0.82
No Own Willingness  9.34±1.24 17.90±1.76 10.29±1.28  8.94±1.24
       No Religion  4.60±0.67 14.45±1.42 13.37±1.07  5.47±0

In [ ]:
"""
COUNTERFACTUAL ANALYSIS — 95% CI + BAR LABELS + WILL-FLIP
===============================================================
Counterfactual conditions included:
  • cf_gdp_flip
  • cf_name_mismatch
  • cf_willingness_flip

Visual Enhancements:
✓ All plots include 95% CI
✓ ΔMAE bars include numeric labels + whiskers
✓ Heatmap shows MAE ± CI
✓ Comparison chart shows CI for all, WILL-FLIP as dot with CI + label

INPUT:
    ablation_original_raw_results.csv
    ablation_structured_climate_raw_results.csv
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style("white")
plt.rcParams.update({'figure.dpi':300,'savefig.dpi':300,'font.size':10,'axes.grid':False})


# ========================================================
# BOOTSTRAP CI
# ========================================================
def CI(sub,n_boot=800):

    if len(sub)==0:
        return{'mae':np.nan,'lo':np.nan,'hi':np.nan}

    mae=mean_absolute_error(sub.actual_other_willingness,sub.prediction)
    boot=[]; idx=np.arange(len(sub)); np.random.seed(42)

    for _ in range(n_boot):
        pick=np.random.choice(idx,len(idx),True)
        boot.append(mean_absolute_error(sub.actual_other_willingness.iloc[pick],
                                        sub.prediction.iloc[pick]))

    return{
        'mae':mae,
        'lo':np.percentile(boot,2.5),
        'hi':np.percentile(boot,97.5)
    }


# ========================================================
# LOAD DATA (KEEP ALL CFs)
# ========================================================
def load_data(file,label):
    print(f"\n📄 Loading {file} [{label}]")
    df=pd.read_csv(file).dropna(subset=['prediction','actual_other_willingness'])
    df.actual_other_willingness*=100; df.actual_own_willingness*=100
    keep=['full','cf_gdp_flip','cf_name_mismatch','cf_willingness_flip']
    df=df[df.condition.isin(keep)]
    print(f"  ✓ {len(df)} rows")
    return df


# ========================================================
# HEATMAP MAE ± CI
# ========================================================
def heatmap(df,stats,out,title):

    models=sorted(df.model.unique())
    conds =['full','cf_gdp_flip','cf_name_mismatch','cf_willingness_flip']
    labels=["Baseline","GDP Flip","Name Mismatch","Will Flip"]

    mat=np.zeros((4,len(models)))*np.nan
    ann=np.empty((4,len(models)),dtype=object)

    for i,c in enumerate(conds):
        for j,m in enumerate(models):
            if (c,m) in stats:
                d=stats[(c,m)]; ci=(d['hi']-d['lo'])/2
                mat[i,j]=d['mae']
                ann[i,j]=f"{d['mae']:.1f}±{ci:.1f}"

    fig,ax=plt.subplots(figsize=(10,7))
    sns.heatmap(mat,annot=ann,fmt="",cmap="RdYlGn_r",
        xticklabels=[m.upper() for m in models],
        yticklabels=labels,cbar_kws={'label':'MAE (pp)'},linewidths=.5)

    ax.set_title(f"{title} — MAE ± 95% CI",fontweight="bold")
    plt.tight_layout(); plt.savefig(out); plt.savefig(out.replace(".png",".pdf")); plt.close()


# ========================================================
# 4-PANEL ΔMAE + CI WHISKERS + LABELS
# ========================================================
def panel(df,stats,base,out,title):

    models=sorted(df.model.unique())
    conds =['cf_gdp_flip','cf_name_mismatch','cf_willingness_flip']
    labs  =["GDP Flip","Name Mismatch","Will Flip"]

    fig,axs=plt.subplots(2,2,figsize=(16,12)); axs=axs.flatten()

    for i,m in enumerate(models):
        ax=axs[i]; vals={}; err=[]

        for c,l in zip(conds,labs):
            d=stats[(c,m)]; b=base[m]['mae']
            mean=d['mae']-b; lo=d['lo']-b; hi=d['hi']-b
            vals[l]=mean; err.append([mean-lo,hi-mean])

        ks=sorted(vals,key=lambda x:vals[x])
        y=np.arange(len(ks)); v=[vals[k] for k in ks]

        bars=ax.barh(y,v,
            color=['#d62728' if x>0 else '#2ca02c' for x in v],
            edgecolor="black",alpha=.75)

        ax.errorbar(v,y,xerr=np.array(err).T,fmt="none",
                    ecolor="black",capsize=4,linewidth=1.2)

        # 🔥 restore numeric labels
        for bar,val in zip(bars,v):
            ax.text(val+(0.3 if val>0 else -0.3),
                    bar.get_y()+bar.get_height()/2,
                    f"{val:+.2f}",
                    ha='left' if val>0 else 'right',
                    va='center',fontweight="bold")

        ax.axvline(0,color="black")
        ax.set_yticks(y); ax.set_yticklabels(ks)
        ax.set_title(f"{m.upper()} baseline {base[m]['mae']:.2f}pp")
        ax.set_xlabel("Δ MAE (pp)")

    plt.suptitle(f"{title} — ΔMAE ±95% CI",fontsize=15,fontweight="bold")
    plt.tight_layout(); plt.savefig(out); plt.savefig(out.replace(".png",".pdf")); plt.close()


# ========================================================
# MULTI-MODEL COMPARISON — WITH CI + NUMERIC LABELS
# ========================================================
def compare(df,stats,base,out,title):

    models=sorted(df.model.unique())
    def get(m,c): return stats[(c,m)]

    b=[base[m]['mae'] for m in models]
    g=[get(m,'cf_gdp_flip')['mae'] for m in models]
    n=[get(m,'cf_name_mismatch')['mae'] for m in models]
    w=[get(m,'cf_willingness_flip')['mae'] for m in models]

    fig,ax=plt.subplots(figsize=(12,7)); x=np.arange(len(models)); W=.25

    for lab,data,col,shift in [
        ("Baseline",[{'mae':h} for h in b],"steelblue",-W),
        ("GDP Flip",[{'mae':h,'lo':stats[('cf_gdp_flip',m)]['lo'],'hi':stats[('cf_gdp_flip',m)]['hi']}
                     for m,h in zip(models,g)],"coral",0),
        ("Name Mismatch",[{'mae':h,'lo':stats[('cf_name_mismatch',m)]['lo'],'hi':stats[('cf_name_mismatch',m)]['hi']}
                          for m,h in zip(models,n)],"indianred",+W)
    ]:
        heights=[d['mae'] for d in data]
        bars=ax.bar(x+shift,heights,W,label=lab,color=col,edgecolor="black")

        # numeric labels
        for bar,d in zip(bars,data):
            h=bar.get_height()
            if np.isfinite(h):
                ax.text(bar.get_x()+bar.get_width()/2,h,
                        f"{h:.2f}",ha="center",va="bottom",fontweight="bold")
                if 'lo' in d:
                    ax.errorbar(bar.get_x()+bar.get_width()/2,h,
                        yerr=[[h-d['lo']],[d['hi']-h]],fmt="none",
                        color="black",capsize=3)

    # 🔥 Will Flip dots + CI + labels
    for i,m in enumerate(models):
        d=stats[('cf_willingness_flip',m)]
        if np.isfinite(d['mae']):
            ax.errorbar(x[i]+W*2,d['mae'],
                yerr=[[d['mae']-d['lo']],[d['hi']-d['mae']]],
                fmt="o",markersize=10,color="gold",mec="black",capsize=4)
            ax.text(x[i]+W*2,d['mae'],f"{d['mae']:.2f}",
                    ha="center",va="bottom",fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in models])
    ax.set_ylabel("MAE (pp)"); ax.set_title(f"{title} — ±95% CI",fontweight="bold")
    ax.legend(frameon=True); plt.tight_layout()

    plt.savefig(out); plt.savefig(out.replace(".png",".pdf")); plt.close()


# ========================================================
# RUN
# ========================================================
def run(file,label,prefix):

    df=load_data(file,label)
    models=sorted(df.model.unique())
    conds =sorted(df.condition.unique())

    base={m:CI(df[(df.model==m)&(df.condition=='full')]) for m in models}
    stats={(c,m):CI(df[(df.model==m)&(df.condition==c)]) for c in conds for m in models}

    panel   (df,stats,base,f"{prefix}_4panel.png",f"{label} — ΔMAE Counterfactuals")
    heatmap (df,stats,f"{prefix}_heatmap.png",f"{label} — MAE")
    compare (df,stats,base,f"{prefix}_compare.png",f"{label} — Baseline vs CF")


if __name__=="__main__":
    run("ablation_original_raw_results.csv","Original","cf_original")
    run("ablation_structured_climate_raw_results.csv","Structured","cf_structured")
    print("\n✔ Counterfactual + Will-Flip CI visualisation READY\n")

In [1]:
"""
COUNTERFACTUAL P-VALUES ANALYSIS
===============================================================
Reports p-values for each counterfactual condition vs. baseline
using bootstrap permutation tests.

INPUT:
    ablation_original_raw_results.csv
    ablation_structured_climate_raw_results.csv

OUTPUT:
    cf_original_pvalues.csv
    cf_structured_pvalues.csv
    Console output with formatted p-value tables
"""

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


# ========================================================
# BOOTSTRAP PERMUTATION TEST FOR P-VALUE
# ========================================================
def permutation_pvalue(sub_baseline, sub_cf, n_perm=2000):
    """
    Two-sided permutation test comparing MAE between conditions.
    H0: MAE_cf = MAE_baseline
    H1: MAE_cf ≠ MAE_baseline
    """
    if len(sub_baseline) == 0 or len(sub_cf) == 0:
        return np.nan
    
    # Observed difference
    mae_baseline = mean_absolute_error(
        sub_baseline.actual_other_willingness, 
        sub_baseline.prediction
    )
    mae_cf = mean_absolute_error(
        sub_cf.actual_other_willingness, 
        sub_cf.prediction
    )
    obs_diff = mae_cf - mae_baseline
    
    # Combine data
    combined_actual = pd.concat([
        sub_baseline.actual_other_willingness,
        sub_cf.actual_other_willingness
    ]).values
    combined_pred = pd.concat([
        sub_baseline.prediction,
        sub_cf.prediction
    ]).values
    
    n_baseline = len(sub_baseline)
    n_total = len(combined_actual)
    
    # Permutation test
    np.random.seed(42)
    perm_diffs = []
    
    for _ in range(n_perm):
        # Shuffle indices
        indices = np.random.permutation(n_total)
        
        # Split into two groups
        idx1 = indices[:n_baseline]
        idx2 = indices[n_baseline:]
        
        # Calculate MAE for each permuted group
        mae1 = mean_absolute_error(combined_actual[idx1], combined_pred[idx1])
        mae2 = mean_absolute_error(combined_actual[idx2], combined_pred[idx2])
        
        perm_diffs.append(mae2 - mae1)
    
    perm_diffs = np.array(perm_diffs)
    
    # Two-sided p-value
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
    
    return {
        'mae_baseline': mae_baseline,
        'mae_cf': mae_cf,
        'delta_mae': obs_diff,
        'p_value': p_value
    }


# ========================================================
# LOAD DATA
# ========================================================
def load_data(file, label):
    print(f"\n📄 Loading {file} [{label}]")
    df = pd.read_csv(file).dropna(subset=['prediction', 'actual_other_willingness'])
    df.actual_other_willingness *= 100
    df.actual_own_willingness *= 100
    keep = ['full', 'cf_gdp_flip', 'cf_name_mismatch', 'cf_willingness_flip']
    df = df[df.condition.isin(keep)]
    print(f"  ✓ {len(df)} rows")
    return df


# ========================================================
# CALCULATE P-VALUES FOR ALL MODELS × CONDITIONS
# ========================================================
def calculate_pvalues(df):
    """
    Calculate p-values for each model comparing baseline to each CF condition
    """
    models = sorted(df.model.unique())
    cf_conditions = ['cf_gdp_flip', 'cf_name_mismatch', 'cf_willingness_flip']
    
    results = []
    
    for model in models:
        print(f"\n🔬 Testing {model.upper()}...")
        
        # Get baseline data for this model
        baseline_data = df[(df.model == model) & (df.condition == 'full')]
        
        for cf_cond in cf_conditions:
            cf_data = df[(df.model == model) & (df.condition == cf_cond)]
            
            print(f"  • {cf_cond}...", end=' ')
            
            test_result = permutation_pvalue(baseline_data, cf_data)
            
            if not np.isnan(test_result['p_value']):
                results.append({
                    'model': model,
                    'condition': cf_cond,
                    'mae_baseline': test_result['mae_baseline'],
                    'mae_cf': test_result['mae_cf'],
                    'delta_mae': test_result['delta_mae'],
                    'p_value': test_result['p_value']
                })
                
                sig = '***' if test_result['p_value'] < 0.001 else \
                      '**' if test_result['p_value'] < 0.01 else \
                      '*' if test_result['p_value'] < 0.05 else 'ns'
                
                print(f"ΔMAE = {test_result['delta_mae']:+.2f}pp, p = {test_result['p_value']:.4f} {sig}")
    
    return pd.DataFrame(results)


# ========================================================
# FORMAT AND DISPLAY RESULTS
# ========================================================
def display_results(results_df, label):
    """
    Display formatted p-value table
    """
    print(f"\n{'='*80}")
    print(f"{label.upper()} — COUNTERFACTUAL P-VALUES")
    print(f"{'='*80}\n")
    
    # Pivot table for easy viewing
    pivot = results_df.pivot_table(
        index='condition',
        columns='model',
        values=['delta_mae', 'p_value']
    )
    
    condition_labels = {
        'cf_gdp_flip': 'GDP Flip',
        'cf_name_mismatch': 'Name Mismatch',
        'cf_willingness_flip': 'Willingness Flip'
    }
    
    for condition in ['cf_gdp_flip', 'cf_name_mismatch', 'cf_willingness_flip']:
        print(f"\n{condition_labels[condition]}")
        print("-" * 80)
        print(f"{'Model':<15} {'MAE_base':<12} {'MAE_CF':<12} {'ΔMAE':<12} {'p-value':<12} {'Sig':<5}")
        print("-" * 80)
        
        cond_data = results_df[results_df.condition == condition]
        
        for _, row in cond_data.iterrows():
            sig = '***' if row['p_value'] < 0.001 else \
                  '**' if row['p_value'] < 0.01 else \
                  '*' if row['p_value'] < 0.05 else 'ns'
            
            print(f"{row['model'].upper():<15} "
                  f"{row['mae_baseline']:>10.2f}pp "
                  f"{row['mae_cf']:>10.2f}pp "
                  f"{row['delta_mae']:>+10.2f}pp "
                  f"{row['p_value']:>10.4f}  "
                  f"{sig:<5}")
    
    print("\n" + "=" * 80)
    print("Significance codes: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
    print("=" * 80 + "\n")


# ========================================================
# RUN ANALYSIS
# ========================================================
def run(file, label, prefix):
    """
    Main analysis pipeline
    """
    df = load_data(file, label)
    results_df = calculate_pvalues(df)
    display_results(results_df, label)
    
    # Save to CSV
    output_file = f"{prefix}_pvalues.csv"
    results_df.to_csv(output_file, index=False)
    print(f"💾 Results saved to {output_file}\n")
    
    return results_df


if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("COUNTERFACTUAL P-VALUE ANALYSIS")
    print("=" * 80)
    
    results_original = run(
        "ablation_original_raw_results.csv",
        "Original",
        "cf_original"
    )
    
    results_structured = run(
        "ablation_structured_climate_raw_results.csv",
        "Structured",
        "cf_structured"
    )
    
    print("\n✔ P-value analysis COMPLETE\n")


COUNTERFACTUAL P-VALUE ANALYSIS

📄 Loading ablation_original_raw_results.csv [Original]
  ✓ 2000 rows

🔬 Testing CLAUDE...
  • cf_gdp_flip... ΔMAE = -0.18pp, p = 0.7215 ns
  • cf_name_mismatch... ΔMAE = +0.35pp, p = 0.5390 ns
  • cf_willingness_flip... ΔMAE = +1.31pp, p = 0.0260 *

🔬 Testing GEMINI...
  • cf_gdp_flip... ΔMAE = -0.82pp, p = 0.4290 ns
  • cf_name_mismatch... ΔMAE = -0.30pp, p = 0.7820 ns
  • cf_willingness_flip... ΔMAE = +3.45pp, p = 0.0005 ***

🔬 Testing GPT...
  • cf_gdp_flip... ΔMAE = +0.17pp, p = 0.8450 ns
  • cf_name_mismatch... ΔMAE = +0.52pp, p = 0.5640 ns
  • cf_willingness_flip... ΔMAE = +4.99pp, p = 0.0000 ***

🔬 Testing LLAMA...
  • cf_gdp_flip... ΔMAE = -0.66pp, p = 0.2835 ns
  • cf_name_mismatch... ΔMAE = -0.33pp, p = 0.5735 ns
  • cf_willingness_flip... ΔMAE = +2.45pp, p = 0.0010 **

ORIGINAL — COUNTERFACTUAL P-VALUES


GDP Flip
--------------------------------------------------------------------------------
Model           MAE_base     MAE_CF       ΔMAE  

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>